# 03 · PyTorch tensors, gradients and your GPU

**Level:** beginner to early intermediate · **Time:** 40–55 minutes · **Data:** tiny synthetic numbers and bundled digits.

You will reason about tensor shapes, move data to a device, calculate gradients, and fit
a straight line using gradient descent. These are the mechanics underneath neural networks.
**Prerequisite:** notebook 01's arrays and basic functions. Everything is recreated here.

Your local GPU can accelerate larger workloads. Tiny examples may run faster on the CPU
because launching GPU work and transferring data also take time.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course_utils.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open Jupyter from the ObjectDetectionPractical folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course_utils import seed_everything, get_device, SMOKE_TEST, ARTIFACTS
seed_everything(42)
device = get_device()
print("Device:", device, "| smoke test:", SMOKE_TEST)


## Step 1 — A tensor is a numeric array with extra abilities
PyTorch tensors have a shape, dtype and device. They can participate in automatic
differentiation and run on compatible accelerators. Float inputs commonly use `float32`;
class labels for cross entropy use integer `int64` (also called `long`).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from course_utils import digit_splits

scalar = torch.tensor(3.0)
vector = torch.tensor([1.0, 2.0, 3.0])
matrix = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
for name, tensor in [("scalar", scalar), ("vector", vector), ("matrix", matrix)]:
    print(name, "shape=", tuple(tensor.shape), "dtype=", tensor.dtype, "device=", tensor.device)


## Step 2 — Convert images and track every dimension
A batch of grayscale images uses `(N, C, H, W)` in PyTorch: examples, channels, height,
width. Our NumPy images have `(N, H, W)`, so `unsqueeze(1)` adds a channel dimension.
`flatten(1)` preserves the batch dimension and flattens the remaining dimensions.


In [ ]:
splits = digit_splits()
train_images, train_labels = splits["train"]
batch_nhw = torch.from_numpy(train_images[:4].copy())
batch_nchw = batch_nhw.unsqueeze(1)
flat_batch = batch_nchw.flatten(1)
batch_labels = torch.from_numpy(train_labels[:4].copy()).long()
print("NHW:", tuple(batch_nhw.shape))
print("NCHW:", tuple(batch_nchw.shape))
print("Flattened:", tuple(flat_batch.shape), "| labels:", batch_labels.tolist())
assert batch_nchw.shape == (4, 1, 8, 8)
assert flat_batch.shape == (4, 64)


## Step 3 — Matrix multiplication creates class scores
`(4, 64) @ (64, 10)` produces `(4, 10)`: ten scores for each of four images.
A bias vector of shape `(10,)` broadcasts across all four rows.
These random scores are **not trained predictions**. Elementwise `*` and matrix `@`
are different operations.


In [ ]:
weights = torch.randn(64, 10) * 0.01
bias = torch.zeros(10)
scores = flat_batch @ weights + bias
print("Class score shape:", tuple(scores.shape))
print("First image scores:", scores[0])
print("Elementwise square:", matrix * matrix)
print("Matrix product:\n", matrix @ matrix)


## Step 4 — Place compatible tensors on the same device
`.to(device)` returns a tensor on that device; assign the returned value.
Model parameters and their inputs must be colocated. `.cpu()` returns data to CPU for
NumPy and plotting. When gradients are involved, `.detach()` first removes the graph connection.


In [ ]:
accelerator_batch = flat_batch.to(device)
accelerator_weights = weights.to(device)
accelerator_bias = bias.to(device)
accelerator_scores = accelerator_batch @ accelerator_weights + accelerator_bias
scores_numpy = accelerator_scores.detach().cpu().numpy()
print("Calculation device:", accelerator_scores.device, "| NumPy shape:", scores_numpy.shape)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("CPU and selected-device results agree:", np.allclose(
    scores_numpy, scores.numpy(), atol=1e-5))


## Step 5 — Ask autograd for a derivative
For `f(w) = (w − 3)²`, the derivative is `2(w − 3)`. At `w=0` it is −6.
A negative derivative means increasing `w` locally reduces this loss.
`requires_grad=True` records operations; `.backward()` computes derivatives and
adds them into each leaf tensor's `.grad` field.


In [ ]:
w = torch.tensor(0.0, requires_grad=True, device=device)
loss = (w - 3.0) ** 2
loss.backward()
print("Loss:", loss.item(), "| gradient:", w.grad.item())
assert np.isclose(w.grad.item(), -6.0)


## Step 6 — Gradients accumulate unless you clear them
A fresh forward calculation creates a fresh graph, but another backward call adds to
existing gradients. Training normally clears gradients each step. The default graph
is freed after backward, so we recompute the loss before another backward call.


In [ ]:
second_loss = (w - 3.0) ** 2
second_loss.backward()
print("After two backward calls:", w.grad.item())
w.grad = None
fresh_loss = (w - 3.0) ** 2
fresh_loss.backward()
print("After clearing and recomputing:", w.grad.item())


## Step 7 — Take gradient-descent steps
Update a parameter with `parameter -= learning_rate * gradient`. The learning rate controls
step size. We update inside `torch.no_grad()` because parameter updates themselves should
not create a new differentiation graph. Watch the parameter approach 3 and the loss approach 0.


In [ ]:
w = torch.tensor(0.0, requires_grad=True, device=device)
learning_rate = 0.1
scalar_history = []
for step in range(25):
    w.grad = None
    loss = (w - 3.0) ** 2
    loss.backward()
    scalar_history.append({"step": step, "w": w.item(), "loss": loss.item()})
    with torch.no_grad():
        w -= learning_rate * w.grad
print("Final parameter:", round(w.item(), 4))
plt.plot([entry["loss"] for entry in scalar_history], marker=".")
plt.xlabel("Update step")
plt.ylabel("Squared distance from 3")
plt.show()


## Step 8 — Fit two parameters to noisy observations
Synthetic targets follow approximately `y = 2x + 1`. The model predicts `a*x+b`;
mean squared error averages the squared prediction errors.
This is a mechanics demonstration, not a generalization experiment: we fit and display
the same tiny training sample. The next notebook restores a proper validation/test protocol.


In [ ]:
seed_everything(42)
x = torch.linspace(-1, 1, 80, device=device).unsqueeze(1)
target = 2.0 * x + 1.0 + 0.08 * torch.randn_like(x)
a = torch.zeros(1, device=device, requires_grad=True)
b = torch.zeros(1, device=device, requires_grad=True)
regression_losses = []

for step in range(120):
    prediction = a * x + b
    loss = ((prediction - target) ** 2).mean()
    loss.backward()
    with torch.no_grad():
        a -= 0.1 * a.grad
        b -= 0.1 * b.grad
    a.grad = None
    b.grad = None
    regression_losses.append(loss.item())

print(f"Learned a={a.item():.3f}, b={b.item():.3f}; generating values were 2 and 1")
with torch.no_grad():
    fitted = a * x + b
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].scatter(x.cpu().numpy(), target.cpu().numpy(), s=12, label="Observations")
axes[0].plot(x.cpu().numpy(), fitted.cpu().numpy(), color="orange", label="Fitted line")
axes[0].legend()
axes[0].set(xlabel="x", ylabel="y")
axes[1].plot(regression_losses)
axes[1].set(xlabel="Step", ylabel="Training mean squared error")
plt.tight_layout()
plt.show()


## Practice 1 — Predict a batch shape
Convert the first seven training images to `(7, 1, 8, 8)` and then to `(7, 64)`.
Write down the output shape before running each operation.

<details><summary>Sample answer</summary>

```python
seven = torch.from_numpy(train_images[:7].copy()).unsqueeze(1)
print(seven.shape, seven.flatten(1).shape)
```

Flattening from dimension 0 would merge examples together, losing the batch boundary.
</details>


In [ ]:
seven_nhw = torch.from_numpy(train_images[:7].copy())
print("Starting shape:", tuple(seven_nhw.shape))
# Add the channel, then flatten only the feature dimensions.


## Practice 2 — Check a derivative by hand
For `f(t)=t³+2t`, calculate the derivative at `t=2`, then check it with autograd.
Which tensor needs `requires_grad=True`?

<details><summary>Sample answer</summary>

```python
t = torch.tensor(2.0, requires_grad=True)
f = t**3 + 2*t
f.backward()
print(t.grad.item())  # 3*(2**2)+2 = 14
```

The input parameter `t` requires gradients; `f` inherits a graph through its operations.
</details>


In [ ]:
trial_value = 2.0
manual_derivative = 3 * trial_value**2 + 2
print("Hand calculation:", manual_derivative)
# Confirm using a fresh tensor and backward().


## Practice 3 — Explore learning-rate behavior
Repeat the scalar `w` experiment with rates `0.01`, `0.1` and `1.1` for 15 steps each.
Record the loss **after** the last update. Which is slow, which converges, which grows?

<details><summary>Sample answer</summary>

```python
for lr in [0.01, 0.1, 1.1]:
    trial = torch.tensor(0.0, requires_grad=True)
    for step in range(15):
        trial.grad = None
        objective = (trial - 3)**2
        objective.backward()
        with torch.no_grad():
            trial -= lr * trial.grad
    print(lr, float((trial.detach() - 3)**2))
```

For this quadratic, `1.1` overshoots and diverges. A useful learning rate depends on the objective.
</details>


In [ ]:
learning_rates_to_try = [0.01, 0.1, 1.1]
print("Compare these rates:", learning_rates_to_try)
# Build independent trial tensors so every run starts at zero.


## What you learned
You tracked tensor dimensions, matrix multiplication and device placement, inspected
`.grad`, cleared accumulated gradients, and used backward passes to learn parameters.
Neural-network layers simply organize larger collections of these differentiable operations.

**Checkpoint:** state the sequence: forward → loss → backward → update, and explain
where gradient clearing belongs.

**Next:** [04 · Your first neural image classifier](04_first_neural_classifier.ipynb).

Primary references: [PyTorch tensors](https://docs.pytorch.org/tutorials/beginner/basics/tensorqs_tutorial.html),
[autograd](https://docs.pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html),
[optimization](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html),
[no_grad](https://docs.pytorch.org/docs/stable/generated/torch.no_grad.html).
